# Topic 42 | Using Context Managers (with Statement)

## The problem
When we open a file, we must close it. If we forget, or if an error happens before `close()` runs, the file stays open. This wastes memory, can lock the file, and may lose unsaved data.

## Context manager
A context manager **sets something up** when a block starts and **cleans it up** when the block ends, automatically, even if an error happens.

We use it with the `with` statement:

    with open("demo.txt", "r") as file:
        data = file.read()

- `open(...)` creates the file object.
- `as file` gives it a name to use inside the block.
- When the block ends, the file is **closed automatically**. You never write `file.close()`.

In [2]:
file = open("demo.txt", "w")
file.write("Hello\n")
file.close()                      # fine here, but we must remember it

# What if an error happens before close()?
file = open("demo.txt", "r")
try:
    data = file.read()
    result = 10 / 0               # error!
    file.close()                  # never reached
except ZeroDivisionError:
    print("Error happened")

print("closed?", file.closed)     # the file is still open
file.close()                      # manual cleanup

Error happened
closed? False


In [3]:
with open("demo.txt", "w") as file:
    file.write("Hello, Python\n")
    file.write("Context managers are great\n")
    print("inside with, closed?", file.closed)

print("after with, closed?", file.closed)

inside with, closed? False
after with, closed? True


In [4]:
try:
    with open("demo.txt", "r") as file:
        print(file.read())
        result = 10 / 0           # error inside the block
except ZeroDivisionError:
    print("Error happened")

print("closed?", file.closed)     # still closed automatically

Hello, Python
Context managers are great

Error happened
closed? True


## Without `with` vs with `with`

| | `open()` + `close()` | `with open()` |
|---|---|---|
| Closes the file | Only if you remember | Always, automatically |
| If an error happens | The file may stay open | The file is still closed |
| Code length | Longer (needs `try/finally` to be safe) | Shorter and cleaner |
| Recommended | No | **Yes** |

## Notes
- The file object is only meant to be used **inside** the `with` block. After it, the file is closed, so reading it gives a `ValueError`.
- The variable (`file`) still exists after the block, but the file is closed.
- You can open several files in one `with` statement (next cell).

In [5]:
with open("demo.txt", "r") as source, open("copy.txt", "w") as target:
    for line in source:
        target.write(line.upper())

with open("copy.txt") as file:
    print(file.read())

HELLO, PYTHON
CONTEXT MANAGERS ARE GREAT



## How it works behind the scenes

Any object can work with `with` if its class has these 2 special methods (you learned special methods in Topic 39):

| Method | Runs | Purpose |
|---|---|---|
| `__enter__` | When the `with` block **starts** | Sets things up. What it returns goes into the `as` variable |
| `__exit__` | When the `with` block **ends**, even after an error | Cleans up (closes the file) |

Python's file object already has both, which is why `with open(...)` works.

In `__exit__(self, exc_type, exc_value, traceback)`, the 3 extra parameters describe an error if one happened. Returning `False` (or nothing) lets the error continue as normal. Returning `True` would hide it.

In [6]:
class FileManager:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        print("Opening file")
        self.file = open(self.filename, self.mode)
        return self.file                      # this goes into "as f"

    def __exit__(self, exc_type, exc_value, traceback):
        print("Closing file")
        self.file.close()
        return False                          # do not hide errors

with FileManager("demo.txt", "r") as f:
    print(f.read())

print(f.closed)

Opening file
Hello, Python
Context managers are great

Closing file
True


In [7]:
from contextlib import contextmanager

@contextmanager
def open_file(name, mode):
    print("Opening file")
    f = open(name, mode)
    try:
        yield f                    # code before yield = __enter__
    finally:
        f.close()                  # code after yield = __exit__
        print("Closing file")

with open_file("demo.txt", "r") as f:
    print(f.readline(), end="")

print(f.closed)

Opening file
Hello, Python
Closing file
True


In [8]:
import time

class Timer:
    def __enter__(self):
        self.start = time.time()
        return self

    def __exit__(self, *args):
        print(f"Took {time.time() - self.start:.1f} seconds")

with Timer():
    time.sleep(0.5)

Took 0.5 seconds


## Where context managers are used
- Files: `with open(...)` (the most common)
- Database connections: open, then always close
- Locks and threads: acquire, then always release
- Timers, temporary settings, test setups

## Key Points
- A context manager does **setup** at the start of a block and **cleanup** at the end.
- `with open(...) as file:` closes the file automatically, even if an error happens.
- Without `with`, you must call `close()` yourself, and an error can skip it.
- A class becomes a context manager by defining `__enter__` and `__exit__`.
- `@contextmanager` from `contextlib` is a shorter way to write one with a function and `yield`.
- Always use `with` for files. It is the standard, recommended way.

## Using Context Managers (other Statements)

`open()` is not the only context manager. Many other tools in Python's standard library work with `with` too, and they all give the same guarantee: **cleanup happens automatically**.

| Tool | What it does |
|---|---|
| `open()` | The standard file context manager for reading and writing files |
| `tempfile.TemporaryFile()` / `NamedTemporaryFile()` | Creates temporary files that **delete themselves** automatically |
| `zipfile.ZipFile()` | Opens `.zip` files for reading and writing |
| `contextlib.ExitStack` | Lets you manage **multiple files** at once easily |
| `pathlib.Path().open()` | Modern, object-oriented file opening through `pathlib` |

In [9]:
import os
import tempfile

# TemporaryFile: no visible name, deleted when closed
with tempfile.TemporaryFile(mode="w+") as temp:
    temp.write("Temporary data")
    temp.seek(0)                      # go back to the start to read
    print(temp.read())

print("closed?", temp.closed)

# NamedTemporaryFile: has a name (temp.name) you can use
with tempfile.NamedTemporaryFile(mode="w+", suffix=".txt") as temp:
    temp.write("Hello from temp")
    temp.flush()                      # make sure the text is written
    name = temp.name
    print("inside, exists?", os.path.exists(name))

print("after, exists?", os.path.exists(name))

Temporary data
closed? True
inside, exists? True
after, exists? False


In [10]:
import zipfile

# First create two small files to zip
with open("a.txt", "w") as f:
    f.write("File A\n")
with open("b.txt", "w") as f:
    f.write("File B\n")

# Write a zip file
with zipfile.ZipFile("archive.zip", "w") as z:
    z.write("a.txt")
    z.write("b.txt")

# Read a zip file
with zipfile.ZipFile("archive.zip", "r") as z:
    print(z.namelist())               # list of files inside
    print(z.read("a.txt").decode())   # read one file's content

['a.txt', 'b.txt']
File A



In [11]:
from contextlib import ExitStack

with open("c.txt", "w") as f:
    f.write("File C\n")

files = ["a.txt", "b.txt", "c.txt"]

with ExitStack() as stack:
    handles = [stack.enter_context(open(name)) for name in files]
    for h in handles:
        print(h.name, "->", h.readline().strip())

print([h.closed for h in handles])

a.txt -> File A
b.txt -> File B
c.txt -> File C
[True, True, True]


In [12]:
from pathlib import Path

p = Path("notes.txt")

with p.open("w") as f:                # same as open("notes.txt", "w")
    f.write("Written with pathlib\n")

with p.open("r") as f:
    print(f.read())

# One-line shortcuts (they open and close the file for you)
p.write_text("Short way\n")
print(p.read_text())

print(p.exists(), p.name, p.suffix)

Written with pathlib

Short way

True notes.txt .txt


## Summary: Which one to use?

| I want to... | Use |
|---|---|
| Read or write a normal file | `with open(...)` |
| Use a scratch file that cleans itself up | `with tempfile.TemporaryFile()` or `NamedTemporaryFile()` |
| Read or create a `.zip` archive | `with zipfile.ZipFile(...)` |
| Open a list of files, any number of them | `with ExitStack() as stack:` |
| Work with file paths in a modern way | `with Path(...).open(...)` |
| Quickly read or write a whole small file | `Path.read_text()` / `Path.write_text()` |

## Key Points
- Many Python tools are context managers, not just `open()`.
- They all follow the same idea: **set up at the start, clean up at the end**, even after an error.
- `tempfile` files delete themselves, so no leftover files.
- `ZipFile` closes the archive automatically.
- `ExitStack` manages a variable number of files with a single `with`.
- `pathlib` is the modern, readable way to work with paths and files.